# Chain-of-Thought & Structured Output

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/building-with-llms/02-chain-of-thought

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Self-consistency by majority vote

A single reasoning chain can slip. If we sample several **independent** chains and the correct answer is more probable than any single wrong one, majority voting boosts accuracy. We simulate a noisy reasoner whose chains are right 55% of the time.

In [ ]:
rng = np.random.default_rng(0)
TRUE = 42
WRONG = [7, 13, 99]

def sample_chain():
    # 55% chance to reach the true answer, else a random wrong one
    return TRUE if rng.random() < 0.55 else int(rng.choice(WRONG))

from collections import Counter
def answer_with_k(k):
    votes = Counter(sample_chain() for _ in range(k))
    return votes.most_common(1)[0][0]

for k in [1, 3, 5, 11]:
    acc = np.mean([answer_with_k(k) == TRUE for _ in range(2000)])
    print(f'k={k:2d} chains -> accuracy {acc:.3f}')

More chains → higher accuracy, with diminishing returns — the cost is running the model `k` times.

## Structured output: parse, don't hope

Downstream code needs machine-readable output. Asking for JSON in prose is fragile; validating against a schema is robust. Here we contrast brittle regex extraction with a schema check.

In [ ]:
import json

good = '{"category": "billing", "confidence": 0.91}'
bad  = 'Sure! Here is the answer: category=billing (91% sure)'

SCHEMA = {'category': str, 'confidence': float}
def validate(text):
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return None
    if all(k in obj and isinstance(obj[k], t) for k, t in SCHEMA.items()):
        return obj
    return None

print('good ->', validate(good))
print('bad  ->', validate(bad))

## ✏️ Your turn

Implement `majority_vote(answers)` returning the most common element (the core of self-consistency).

In [ ]:
def majority_vote(answers):
    # TODO(you): return the value that appears most often in `answers`.
    return None

assert majority_vote([42, 7, 42, 99, 42]) == 42
assert majority_vote(['a', 'b', 'b']) == 'b'
print('passed ✓')

<details><summary>Solution</summary>

```python
from collections import Counter
def majority_vote(answers):
    return Counter(answers).most_common(1)[0][0]
```

</details>